# KEGG annotation and DEG enrichment analysis

This notebook contains two consecutive stages:

1. generation of the gene-level KEGG annotation for *Leptinotarsa decemlineata*;
2. KEGG over-representation analysis of differential expression results generated in Step 1.

In [ ]:
import pandas as pd
import numpy as np
import json
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import fdrcorrection as fdr


## 1. Prepare the Ldec_2.0 gene annotation used for the KEGG merge

`ldec_gff_annotated.tsv` is generated by `00_prepare_expression_inputs.ipynb`.

In [ ]:
ldec_gtf2 = pd.read_csv('ldec_gff_annotated.tsv', sep='\t')

## gen_anno <- unique.data.frame( gff_anno[,c('gene_id', 'product')] )
gen_anno = ldec_gtf2.loc[:, ['gene_id', 'gene_name', 'product']].drop_duplicates()

#gen_anno['product'].loc[
gen_anno['product_fxd'] = gen_anno['product'].fillna('').str.replace('%2C', ',')

gen_anno


## 2. Parse the KEGG hierarchy and generate the gene-level KEGG annotation


In [ ]:
with open('./fromKEGG/ldc00001.json') as f:
    ldc_paths = json.load(f)

In [ ]:
from flatten_json import flatten, unflatten, unflatten_list

In [ ]:
flat = flatten(ldc_paths)
unflat = unflatten ( flat )

In [ ]:
#flat

In [ ]:
o = open('ldec_hierarch_kegg.tsv', 'w')
for k in flat.keys():
    string = (len(k.split('_')) - 2)*'\t'+flat[k].replace('\t', ' ')
    ext = (8-len(string.split('\t')))*'\t'
    o.write( string+ext +"\n")
o.close()

In [ ]:
mykegg = pd.read_table('ldec_hierarch_kegg.tsv', sep = '\t', header = None ).fillna(method='ffill', axis = 0)

In [ ]:
#mykegg

In [ ]:
mykegg.dropna(axis=1, how='all').dropna(axis=0).drop(columns = [0]).to_csv('my_ldec_kegg_anno.tsv', sep = '\t', 
                                                                           index = False)

In [ ]:
mykegg = pd.read_table('my_ldec_kegg_anno.tsv', sep = '\t' )

In [ ]:
#mykegg

In [ ]:
mykegg.columns = ['general', 'group', 'path', 'gene']

In [ ]:
#mykegg.general.value_counts()

In [ ]:
## to exclude:
## 09190 Not Included in Pathway or Brite         235
## 09160 Human Diseases                           112
mykegg_f = mykegg.loc[ ~ ( mykegg.general.str.contains('09190 Not Included in Pathway or Brite')| \
                           mykegg.general.str.contains('09160 Human Diseases') ) ]

In [ ]:
#mykegg_f

In [ ]:
## get gene names:
mykegg['gene_id'] = 'gene-LOC' + mykegg.gene.str.extract('^(\d{6,})')

In [ ]:
## extract KEGG known orthologs
mykegg['KO'] = mykegg.gene.str.extract('(K\d{4,})')

In [ ]:
## extract EC codes
mykegg['EC'] = mykegg.gene.str.extract('\[EC:(.+)\]').fillna('').iloc[:, 0].str.split()

In [ ]:
## extract Brite Hierarchies
mykegg['BR'] = mykegg.path.str.extract('\[BR:(.+)\]').fillna('')

In [ ]:
## extract Brite Hierarchies
mykegg['KEGG_PATH'] = mykegg.path.str.extract('\[PATH:(.+)\]').fillna('')

In [ ]:
mykegg['general_id'] = mykegg.general.str.extract('^(\d{5,})')

In [ ]:
mykegg['group_id'] = mykegg.group.str.extract('^(\d{5,})')

In [ ]:
mykegg['path_id'] = mykegg.path.str.extract('^(\d{5,})')

In [ ]:
mykegg_e = mykegg.explode('EC')

In [ ]:
mykegg_e['EC'] = ('EC:' + mykegg_e['EC']).fillna('')

In [ ]:
mykegg_e2 = mykegg_e.merge( gen_anno, how = 'outer' )

In [ ]:
mykegg_e2['general'] = mykegg_e2.general.str.replace('^\d{5,} ', '', regex = True)

In [ ]:
mykegg_e2['group'] = mykegg_e2.group.str.replace('^\d{5,} ', '', regex = True)

In [ ]:
mykegg_e2['path'] = mykegg_e2.path.str.replace('^\d{5,} ', '', regex = True).str.replace(' \[.+?\]', '', regex = True)

In [ ]:
mykegg_e2_annotated = mykegg_e2[~mykegg_e2.gene_name.isna()]

In [ ]:
mykegg_e2_annotated['EC3'] = mykegg_e2_annotated.EC.apply(lambda x: '.'.join( x.split('.')[:-1]) )

In [ ]:
mykegg_e2_annotated['EC2'] = mykegg_e2_annotated.EC.apply(lambda x: '.'.join( x.split('.')[:-2]) )

In [ ]:
mykegg_e2_annotated['EC1'] = mykegg_e2_annotated.EC.apply(lambda x: '.'.join( x.split('.')[:-3]) )

In [ ]:
mykegg_e2_annotated.to_csv('ldec_genes_annotated_kegg.tsv', sep = '\t', index = False)

In [ ]:
mykegg_e2_annotated.to_excel('ldec_genes_annotated_kegg.xlsx', index = False)


## 3. KEGG enrichment analysis of differentially expressed genes

The enrichment analysis uses the KEGG annotation generated above together with DESeq2 result tables from Step 1.
Inputs from Step 1:
- `DEG_ldec_DESeq2.xlsx`
- `DEG_ldec_DESeq2_bbas_vs_mrob.xlsx`

#### Observed
|      |in list|not in list|totals|
| :-   | --:   |      --:  | --:  |
|with annotation| A| B| A+B|
|without annotation| C| D| C+D|
| |A+C| B+D| A+B+C+D=N|
  
  
#### Expected
|      |in list|not in list|totals|
| :-   | --:   |      --:  | --:  |
|with annotation| (A+B)(A+C)/N| (A+B)(B+D)/N| A+B|
|without annotation| (C+D)(A+C)/N| (C+D)(B+D)/N| C+D|
| |A+C| B+D| A+B+C+D=N|

In [ ]:
test_set = pd.read_table('./test_ldec_diffexp.txt', header = None).iloc[:, 0]

In [ ]:
genes_in = test_set.values[:200]
genes_out = test_set.values[200:]
#mykegg_e2_annotated.gene_id

In [ ]:
def enrich( genes_to_test, column = 'KEGG_PATH', anno = mykegg_e2_annotated, topN = 100 , padj = True):
    
    genes_in  = genes_to_test.values[:topN]
    genes_out = genes_to_test.values[topN:]
    
    inlist  = anno.loc[
        anno.gene_id.isin(genes_in) & \
        ~anno[column].isna() & \
        ~anno[column].eq('') ][['gene_id', column]].drop_duplicates()
    inlist_ext  = anno.loc[
        anno.gene_id.isin(genes_in) & \
        ~anno[column].isna() & \
        ~anno[column].eq('') ][['gene_id', 'product_fxd', 'path', column]].drop_duplicates()
    outlist = anno.loc[
        anno.gene_id.isin(genes_out) & \
        ~anno[column].isna() & \
        ~anno[column].eq('') ][['gene_id', column]].drop_duplicates()
    
    ac = inlist.gene_id.nunique()
    bd = outlist.gene_id.nunique()
    
    L = []
    
    for ID in inlist[column].unique():
        a, b = (inlist[column] == ID).sum(), (outlist[column] == ID).sum() 
        c = ac-a
        d = bd-b
        #print (ID, a, b, c, d)
        OR, p = fisher_exact(np.array([[a,b],[c,d]]), alternative = 'greater')
        L += [pd.Series ({column:ID, 'a':a, 'b':b, 'c':c, 'd':d, 'OR':OR, 'p':p,
                          'gene_ids':'; '.join(inlist[inlist[column] == ID].gene_id.unique()),
                          'genes':' & '.join(inlist_ext[inlist_ext[column] == ID].product_fxd.unique()),
                          'descr_path':'; '.join(inlist_ext[inlist_ext[column] == ID].path.unique())})]
    L = pd.concat (L, axis = 1).T
    L = L.sort_values('p').reset_index(drop = True)
    if padj:
        L['fdr'] = fdr(L['p'].values)[1]
    else:
        L['fdr'] = np.nan
    return L[[column, 'a', 'b', 'c', 'd', 'OR', 'p', 'fdr', 'descr_path', 'gene_ids', 'genes']]

In [ ]:
#enrich(test_set, column = 'group', topN=100)
enrich(test_set, column = 'EC3', topN=50).head(30)

In [ ]:
enrich(test_set, column = 'EC3', topN=50).head(30)

In [ ]:
enrich(test_set, column='general', topN=500)

#### Read DEG table from big xlsx workbook

In [ ]:
deg = pd.read_excel('./DEG_ldec_DESeq2.xlsx', sheet_name = None)

In [ ]:
deg.keys()

```
Bbas_All	Differential expression - B.bassiana vs. control
Mrob_All	Differential expression - M.robertsii vs. control
Fat_vs_Hae_All	Differential expression - Haemocytes vs. FatBody
		
    Haemocytes and fat body samples were also analysed separately	
Bbas_Hae	Differential expression - B.bassiana vs. control
Mrob_Hae	Differential expression - M.robertsii vs. control
		
    FatBody - Differential expression was tested with DESeq2 using all fat body samples
Bbas_Fat	Differential expression - B.bassiana vs. control
Mrob_Fat	Differential expression - M.robertsii vs. control
		
    FatBody_woC17 - Differential expression was tested with DESeq2 using fat body samples 
Bbas_Fat_woC17	Differential expression - B.bassiana vs. control
Mrob_Fat_woC17	Differential expression - M.robertsii vs. control
```

In [ ]:
sel_keys = ["Bbas_All", "Mrob_All", "Fat_vs_Hae_All", "Bbas_Hae", "Mrob_Hae", 
            "Bbas_Fat", "Mrob_Fat", "Bbas_Fat_woC17", "Mrob_Fat_woC17"]

In [ ]:
## for key in sel_keys:
##     for column in ['KEGG_PATH', 'KO', 'BR', 'EC', 'EC3', 'group']:
##         for n in [50, 150, 200, 500]

In [ ]:
#enrich(deg[sel_keys[0]].gene_id, column = 'KO', topN=500).head(30)


In [ ]:
for key in sel_keys:
    for column in ['KEGG_PATH', 'KO', 'BR', 'EC', 'EC3', 'group']:
        writer = pd.ExcelWriter(f"./{key}_{column}_enrichment.xlsx", engine="openpyxl", mode="w")
        for n in [50, 150, 200, 500]:
            tmp = enrich( deg[key].gene_id , column = column, topN=n)
            tmp.to_excel(writer, sheet_name = f'{column}_top{n}')
        writer.close()
            


In [ ]:
deg2 = pd.read_excel('./DEG_ldec_DESeq2_bbas_vs_mrob.xlsx', sheet_name = None)

In [ ]:
for key in deg2.keys():
    for column in ['KEGG_PATH', 'KO', 'BR', 'EC', 'EC3', 'group']:
        writer = pd.ExcelWriter(f"./{key}_{column}_enrichment.xlsx", engine="openpyxl", mode="w")
        for n in [50, 150, 200, 500]:
            tmp = enrich( deg2[key].gene_id , column = column, topN=n)
            tmp.to_excel(writer, sheet_name = f'{column}_top{n}')
        writer.close()

In [ ]:
#deg2.keys()

In [ ]:
(deg2[key]['padj'] < 0.05).sum()

In [ ]:
for key in sel_keys:
    writer = pd.ExcelWriter(f"./{key}_DEG_KEGG_enrichment.xlsx", engine="openpyxl", mode="w")
    for column in ['KEGG_PATH', 'KO', 'BR', 'EC', 'EC3', 'group']:
        n = (deg[key]['padj'] < 0.05).sum()
        tmp = enrich( deg[key].gene_id , column = column, topN=n)
        tmp.to_excel(writer, sheet_name = f'{column}_DEG{n}')
    writer.close()

In [ ]:
for key in deg2.keys():
    writer = pd.ExcelWriter(f"./{key}_DEG_KEGG_enrichment.xlsx", engine="openpyxl", mode="w")
    for column in ['KEGG_PATH', 'KO', 'BR', 'EC', 'EC3', 'group']:
        n = (deg2[key]['padj'] < 0.05).sum()
        tmp = enrich( deg2[key].gene_id , column = column, topN=n)
        tmp.to_excel(writer, sheet_name = f'{column}_DEG{n}')
    writer.close()